## FCT 자격증 f_hr_license
--------------------------------------
 - 화이트 + 더존 각각 유니온된 상태
    - 화이트와 더존은 서로 각기 다른 로직을 사용하기 때문에 문제가 생기면 화이트는 상위쿼리에서 더존은 하위 쿼리에서 각각 수정해야함
    - 화이트를 기준으로 컬럼을 생성했기 때문에 화이트엔 있는 컬럼이지만 더존에는 없을수도 있음 -> 그런 경우 더존은 NULL처리

 - 더존은 EMP_NO만 존재, EMP_ID는 존재하지않기 때문에 EMP_ID >  알파벳 + EMP_NO 으로 구성함 즉, EMP_ID와 EMP_NO는 같은 데이터

 - 대시보드에 추가하기 위해서 컬럼추가시, 더존 & 화이트 모두 추가해야 수정가능함.
   - UNION을 기준으로  쿼리 컬럼수와 순서가 일치하지않으면 테이블 수정 불가능
   - 컬럼수와 순서를 맞췄는데 오류가 나면 CAST도 확인해야함 -> 화이트는 대부분 timestamp 형태, 더존은 text형태로 CAST 구문 추가해야함
   - 날짜형식은 바꾸면 데이터가 안나오는 경우가 많으므로, CAST 후, 데이터 나오는지 확인 필수!

In [ ]:
df = spark.sql("""
-- 화이트 쿼리               
SELECT   
LICENSE.MOD_USER_ID  AS MOD_PRSN,  
LICENSE.MOD_DATE AS MOD_DTT,  
LICENSE.TZ_CD AS TZ_CD,  
LICENSE.TZ_DATE AS TZ_DTT,  
CAST(LICENSE.EMP_ID AS string ) AS EMP_ID,  
LICENSE.PHM_LICENSE_ID AS EMP_LCNS_ID,  
LICENSE.LICENSE_COMPANY_NM AS ISSUE_ORG,  
LICENSE.LICENSE_TYPE_CD AS LCNS_DIV_CD,  
COMM_LICENSE_TY.CD_NM AS LCNS_DIV_NM,  
COMM_LICENSE.CD_NM AS LCNS_NM,  
LICENSE.NOTE AS NOTE,  
LICENSE.BONUS_TYPE AS BONUS_PAY_DIV_CD,  
LICENSE.REG_YN AS REG_YN,  
LICENSE.LICENSE_NO AS LCNS_NO,  
to_timestamp(LICENSE.END_YMD , 'yyyyMMdd')AS VALID_DT,  
to_timestamp(LICENSE.STA_YMD , 'yyyyMMdd') AS GET_DT,  
LICENSE.PERSON_ID  AS PRSNL_ID,  
LICENSE.LICENSE_CD AS LCNS_CD  
FROM brz.white.PHM_LICENSE LICENSE  
LEFT JOIN brz.white.VE_FRM_CODE COMM_LICENSE ON COMM_LICENSE.CD_KIND ='PHM_LICENSE_CD' AND LICENSE.LICENSE_CD = COMM_LICENSE.CD  
LEFT JOIN brz.white.VE_FRM_CODE COMM_LICENSE_TY ON COMM_LICENSE_TY.CD_KIND ='PHM_LICENSE_TYPE_CD' AND LICENSE.LICENSE_TYPE_CD = COMM_LICENSE_TY.CD  
  
UNION  
-- 더존 쿼리
SELECT 
NULL as MOD_PRSN,  
l.UPDATE_DTS as MOD_DTT,  
NULL as TZ_CD,  
NULL as TZ_DTT,   
CASE  
 WHEN l.COMPANY_CD = '3000' THEN 'a' || l.EMP_NO -- Company A
 WHEN l.COMPANY_CD = '2000' THEN 'q' || l.EMP_NO -- Company B
 WHEN l.COMPANY_CD = '1000' THEN 'f' || l.EMP_NO -- Company C
  ELSE l.EMP_NO  
 END AS EMP_ID,
NULL as EMP_LCNS_ID,  
l.ISSUE_ISTN_NM as ISSUE_ORG,  
NULL as LCNS_DIV_CD,  
NULL as LCNS_DIV_NM,  
l.QUA_CARD_NM as LCNS_NM,  
l.RMK_DC as NOTE,  
NULL as BONUS_PAY_DIV_CD,  
NULL as REG_YN,  
NULL as LCNS_NO,  
to_timestamp(l.VLID_DT , 'yyyyMMdd') as VALID_DT,  
to_timestamp(l.ACQS_DT , 'yyyyMMdd') as GET_DT,  
NULL as PRSNL_ID,  
NULL as LCNS_CD  
FROM slv.dzn.hr_license_mst l  
WHERE l.COMPANY_CD in ('1000', '2000', '3000') 
""")

In [ ]:
# gld 쌓기 
df.write.mode("overwrite").save("abfss://gld@<storage-account>.dfs.core.windows.net/f_hr_license");

In [ ]:
%%sql
-- gld에 쌓인 데이터 브릭스 테이블 안에 쌓기
USE gld.default;
CREATE TABLE IF NOT EXISTS f_hr_license
USING DELTA
LOCATION 'abfss://gld@<storage-account>.dfs.core.windows.net/f_hr_license';

In [ ]:
%%sql
select count(*) from gld.default.f_hr_license